<a href="https://colab.research.google.com/github/mahsa-github/mahsa-github/blob/main/Getting_Started_With_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting started with Layer

[![Open in Layer](https://app.layer.ai/assets/badge.svg)](https://app.layer.ai/layer/titanic) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/layerai/examples/blob/main/titanic/Getting_Started_With_Layer.ipynb) [![Layer Examples Github](https://badgen.net/badge/icon/github?icon=github&label)](https://github.com/layerai/examples/tree/main/titanic)

In this quick walkthrough, we will train a machine learning model to predict the survivors of the Titanic disaster and deploy it for real-time inference using Layer.


## Installation


In [2]:
pip install scikit-learn numpy pandas

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [3]:
!pip install -U layer -q

     |████████████████████████████████| 192 kB 5.9 MB/s 
     |████████████████████████████████| 4.7 MB 41.5 MB/s 
     |████████████████████████████████| 106 kB 45.5 MB/s 
     |████████████████████████████████| 4.1 MB 45.0 MB/s 
     |████████████████████████████████| 235 kB 48.6 MB/s 
     |████████████████████████████████| 26.7 MB 2.0 MB/s 
     |████████████████████████████████| 132 kB 79.5 MB/s 
     |████████████████████████████████| 1.3 MB 55.6 MB/s 
     |████████████████████████████████| 341 kB 71.6 MB/s 
     |████████████████████████████████| 3.5 MB 59.2 MB/s 
     |████████████████████████████████| 2.4 MB 46.4 MB/s 
     |████████████████████████████████| 79 kB 8.9 MB/s 
     |████████████████████████████████| 9.0 MB 47.7 MB/s 
     |████████████████████████████████| 139 kB 78.5 MB/s 
     |████████████████████████████████| 77 kB 6.6 MB/s 
     |████████████████████████████████| 181 kB 57.8 MB/s 
     |████████████████████████████████| 596 kB 57.8 MB/s 
     |█████████████

In [4]:
from layer.decorators import dataset, model,resources
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import layer
from layer import Dataset

## Register and Login

To start using Layer, you have to register and login. Run the following cell, click the link, register and paste the code in the input

In [4]:
layer.login()

Please open the following link in your web browser. Once logged in, copy the code and paste it here.
https://app.layer.ai/oauth/authorize?response_type=code&code_challenge=HWyP9Om3eJrPKENLnVVqOoVnKiOm5XDHrcxhCM5jZXc&code_challenge_method=S256&client_id=0STDdcnpK48P8A429EAAn93WNuLmViLR&redirect_uri=https://app.layer.ai/oauth/code&scope=offline_access&audience=https://app.layer.ai
Code: B6IvWsJmVU06k3IuNNrRyseAzSg2YMr0dk83I8krky40r
Successfully logged into https://app.layer.ai


## Inititialize Your First Layer Project

It's time to create your first Layer Project. You can find your created project at [https://app.layer.ai](https://app.layer.ai)

In [82]:
sample_submission = layer.get_dataset("layer/Zindi-Laduma-Analytics/datasets/sample-submission").to_pandas()
sample_submission.head()

Output()

,Game_ID,Away win,Draw,Home Win
0,ID_8518U587,0,0,0
1,ID_H49BIKG7,0,0,0
2,ID_PO6SP4VA,0,0,0
3,ID_MZRCNBAQ,0,0,0
4,ID_CV9VOLIU,0,0,0


In [156]:
train = layer.get_dataset("layer/Zindi-Laduma-Analytics/datasets/train").to_pandas()
train.head()


train_game_statistics = layer.get_dataset("layer/Zindi-Laduma-Analytics/datasets/train_game_statistics").to_pandas()


Output()

Output()

In [161]:
total_train = train.merge(train_game_statistics, how='inner', on='Game_ID')

In [162]:
total_train.sort_values("Date", inplace= True, ignore_index= True)

In [163]:
test = layer.get_dataset("layer/Zindi-Laduma-Analytics/datasets/test").to_pandas()
test_game_statistics = layer.get_dataset("layer/Zindi-Laduma-Analytics/datasets/test_game_statistics").to_pandas()

Output()

Output()

In [105]:
total_test = test.merge(test_game_statistics, how='inner', on='Game_ID')

In [164]:
col = ['next_player', 'next_action', 'next_x','next_y', 'event_id', 'next_team', 'next_event_id', 'xt_value']
total_train.drop(columns= col, inplace= True)

In [165]:
total_train['mins'] = total_train['End_minutes']-total_train['Start_minutes']
total_train.drop(columns= ['End_minutes','Start_minutes'], inplace= True)

In [166]:
bin_features = ['Shots','Season_x','Half','Season_y','SoT','Goals_scored','Goals_conceded','Accurate passes','Inaccurate passes','Passes']
cat_features = ['Home Team','Away Team','Team','Action','Manager','Opposition_Team']

In [167]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [168]:
total_train['Shots'] = le.fit_transform(total_train['Shots'])
total_train['Season_x']= le.fit_transform(total_train['Season_x'])
total_train['Half']= le.fit_transform(total_train['Half'])
total_train['Season_y']= le.fit_transform(total_train['Season_y'])
total_train['SoT']= le.fit_transform(total_train['SoT'])
total_train['Goals_scored']= le.fit_transform(total_train['Goals_scored'])
total_train['Goals_conceded']= le.fit_transform(total_train['Goals_conceded'])
total_train['Accurate passes']= le.fit_transform(total_train['Accurate passes'])
total_train['Inaccurate passes']= le.fit_transform(total_train['Inaccurate passes'])
total_train['Passes']= le.fit_transform(total_train['Passes'])

In [169]:
total_train = pd.get_dummies(total_train, columns= cat_features,drop_first= True)

In [175]:
total_train.head(3)

,Date,Season_x,Match_ID,Game_ID,Score,Player_ID,id,X,Y,Half,...,Opposition_Team_Cosmos Redshift 7,Opposition_Team_Eye of Sauron,Opposition_Team_Fireworks,Opposition_Team_Medusa Merger,Opposition_Team_Milky Way,Opposition_Team_Sculptor,Opposition_Team_Sombrero,Opposition_Team_Sunflower,Opposition_Team_Tadpole,Opposition_Team_Triangulum
0,2016-08-03,0,77,ID_IDJ1L6GL,Draw,Player_C20PB9QR,1,52,33,0,...,0,0,1,0,0,0,0,0,0,0
1,2016-08-03,0,158,ID_WT3PHABY,Home Win,Player_GY2VJNI8,2846,32,61,1,...,0,0,0,0,1,0,0,0,0,0
2,2016-08-03,0,158,ID_WT3PHABY,Home Win,Player_6OGDFTPQ,2847,46,62,1,...,0,0,0,0,1,0,0,0,0,0


In [173]:
total_train['id']= total_train['id'].astype('int')
total_train['mins']= total_train['mins'].astype('int')
total_train['X'] = total_train['X'].astype('int')
total_train['Y'] = total_train['Y'].astype('int')
total_train['Match_ID'] = total_train['Match_ID'].astype('int')

Baseline Model

0

In [125]:
total_train['Passes'].value_counts()

0.0    1194984
1.0     376593
Name: Passes, dtype: int64

In [113]:
total_train.columns

Index(['Date', 'Season_x', 'Match_ID', 'Game_ID', 'Home Team', 'Away Team',
       'Score', 'Player_ID', 'id', 'X', 'Y', 'Team', 'Action', 'Half',
       'Season_y', 'Manager', 'Opposition_Team', 'Shots', 'SoT',
       'Goals_scored', 'Goals_conceded', 'Accurate passes',
       'Inaccurate passes', 'Passes', 'mins'],
      dtype='object')

In [16]:
key = train.Game_ID[0]
print(key)

ID_KAG4KAE9


In [17]:
print(train.get(key))

None


In [8]:
bazi = train.Match_ID[train.Match_ID == 169.0].index.tolist()

In [9]:
bazi

[229, 302, 388]

In [10]:
train[train.Match_ID == 169.0]

,Date,Season,Match_ID,Game_ID,Home Team,Away Team,Score
229,2018-04-27,2,169.0,ID_KC53ZIHC,Fireworks,Comet,Away win
302,2017-08-15,1,169.0,ID_MWHDDOC0,Eye of Sauron,Cigar,Away win
388,2017-01-19,1,169.0,ID_PBG2XS57,Milky Way,Medusa Merger,Away win


In [34]:
train['Date']= pd.to_datetime(train['Date'])


In [58]:
train_game_statistics['next_team'].isna().sum()

1571577

In [42]:
total_train["min"]= total_train["End_minutes"]- total_train["Start_minutes"]
total_train.drop(columns= ["End_minutes","Start_minutes"], inplace= True)

In [70]:
#total_train.drop(columns= col, inplace= True)

In [69]:
total_train['Shots']= total_train['Shots'].astype('int')
total_train['id']= total_train['id'].astype('int')
total_train['xy']= total_train['X']*total_train['Y'].astype('int')
total_train['Goals_differ']= total_train['Goals_scored']+total_train['Goals_conceded'].astype('int')
total_train['Accurate passes']= total_train['Accurate passes'].astype('int')
total_train['Inaccurate passes']= total_train['Inaccurate passes'].astype('int')
total_train['Passes']= total_train['Passes'].astype('int')
total_train['min']= total_train['min'].astype('int')

In [72]:
col2 = ['X', 'Y','Goals_scored','Goals_conceded']
total_train.drop( columns= col2, inplace= True)

In [78]:
total_train['xy'] = total_train['xy'].astype('int')
total_train['Goals_differ'] = total_train['Goals_differ'].astype('int')
total_train['Match_ID'] = total_train['Match_ID'].astype('int')
total_train['SoT'] = total_train['SoT'].astype('int')

In [79]:
total_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1571577 entries, 1228324 to 907789
Data columns (total 21 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   Date               1571577 non-null  datetime64[ns]
 1   Season_x           1571577 non-null  int64         
 2   Match_ID           1571577 non-null  int64         
 3   Home Team          1571577 non-null  object        
 4   Away Team          1571577 non-null  object        
 5   Player_ID          1571577 non-null  object        
 6   id                 1571577 non-null  int64         
 7   Team               1571577 non-null  object        
 8   Action             1571577 non-null  object        
 9   Half               1571577 non-null  object        
 10  Season_y           1571577 non-null  int64         
 11  Manager            1571577 non-null  object        
 12  Opposition_Team    1571577 non-null  object        
 13  Shots              157

In [22]:
test = layer.get_dataset("layer/Zindi-Laduma-Analytics/datasets/test").to_pandas()
test_game_statistics = layer.get_dataset("layer/Zindi-Laduma-Analytics/datasets/test_game_statistics").to_pandas()

Output()

Output()

In [80]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       234 non-null    object 
 1   Season     234 non-null    int64  
 2   Match_ID   234 non-null    float64
 3   Game_ID    234 non-null    object 
 4   Home Team  234 non-null    object 
 5   Away Team  234 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 11.1+ KB


In [81]:
test_game_statistics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 780234 entries, 0 to 780233
Data columns (total 24 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Game_ID            780234 non-null  object 
 1   Player_ID          780234 non-null  object 
 2   id                 780234 non-null  float64
 3   X                  780234 non-null  float64
 4   Y                  780234 non-null  float64
 5   Team               780234 non-null  object 
 6   Half               780223 non-null  object 
 7   Season             780234 non-null  int64  
 8   Manager            774632 non-null  object 
 9   Opposition_Team    780234 non-null  object 
 10  Shots              780142 non-null  float64
 11  SoT                780142 non-null  float64
 12  Accurate passes    780142 non-null  float64
 13  Inaccurate passes  780142 non-null  float64
 14  Passes             780142 non-null  float64
 15  Start_minutes      780224 non-null  float64
 16  En

## Where to go from here?

Now that you have created first Layer Project, you can:

- Join our [Slack Community ](https://bit.ly/layercommunityslack)
- Visit [Layer Examples Repo](https://github.com/layerai/examples) for more examples
- Browse [Trending Layer Projects](https://layer.ai) on our mainpage
- Check out [Layer Documentation](https://docs.layer.ai) to learn more